# Memory Architecture: routing memory to where it belongs

An agent's memory is not one thing. A user's preferences, an organization's compliance
rules, and a library of reusable skills are all just files to the agent &mdash; but they have
completely different lifetimes, owners, and permissions. Preferences belong to **one user**.
Policies belong to **the organization** and the agent must never edit them. Scratch work should
evaporate when the thread ends.

<code style="font-size:15px; color:#2b8a3e;">CompositeBackend</code> is how you get all of
that from a single filesystem: it is a **path-prefix router**. The agent sees one flat tree
and calls the same `read_file` / `edit_file` tools it always does. Underneath, each prefix
goes to a different backend with a different persistence scope.

This notebook builds one agent with three memory types and proves each property &mdash;
isolation, read-only enforcement, and durability across threads &mdash; by observing the
store directly rather than taking the model's word for it.

<table style="font-size:16px; border-collapse:collapse; width:100%;">
<thead>
<tr>
<th style="border:1px solid #999; padding:10px; text-align:left;">Path</th>
<th style="border:1px solid #999; padding:10px; text-align:left;">Backend</th>
<th style="border:1px solid #999; padding:10px; text-align:left;">Scope</th>
<th style="border:1px solid #999; padding:10px; text-align:left;">Agent can write?</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #999; padding:10px;"><code style="color:#2b8a3e;">/memories/</code></td>
<td style="border:1px solid #999; padding:10px;"><code style="color:#2b8a3e;">StoreBackend</code></td>
<td style="border:1px solid #999; padding:10px;">per user</td>
<td style="border:1px solid #999; padding:10px;">Yes</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px;"><code style="color:#2b8a3e;">/policies/</code></td>
<td style="border:1px solid #999; padding:10px;"><code style="color:#2b8a3e;">StoreBackend</code></td>
<td style="border:1px solid #999; padding:10px;">per org</td>
<td style="border:1px solid #999; padding:10px;"><b>No</b> &mdash; denied by permission</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px;"><code style="color:#2b8a3e;">/skills/</code></td>
<td style="border:1px solid #999; padding:10px;"><code style="color:#2b8a3e;">StoreBackend</code></td>
<td style="border:1px solid #999; padding:10px;">per user</td>
<td style="border:1px solid #999; padding:10px;">Yes</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px;">everything else</td>
<td style="border:1px solid #999; padding:10px;"><code style="color:#2b8a3e;">StateBackend</code></td>
<td style="border:1px solid #999; padding:10px;">per thread</td>
<td style="border:1px solid #999; padding:10px;">Yes, but ephemeral</td>
</tr>
</tbody>
</table>

> There is no memory *classifier* here. Nothing inspects a write and decides "this is
> semantic memory." The routing is pure string-prefix matching, and the meaning of each
> prefix lives entirely in your path design and your system prompt.

The other half of the picture is *what the model actually sees*. Only some of this reaches
the prompt: `memory=[…]` files are pasted in whole on every call, `skills=[…]` contributes
a one-line index and nothing more, and everything else stays on disk until a tool goes and
fetches it.

<img src="images/deep-agent-memory-regions.svg"
     alt="Two columns. Left: the system message the model sees on every call — the authored system_prompt, a skills index appended by SkillsMiddleware carrying only each skill's name and description, and an agent_memory block appended by MemoryMiddleware holding the full text of every memory= file. Right: the virtual filesystem, where /skills/, /memories/ and /policies/ route to durable per-user and per-org store namespaces and everything else falls through to thread-scoped state. Solid arrows mark content loaded into the prompt automatically; dashed arrows mark content fetched only when the model calls a filesystem tool."
     style="width:100%; height:auto; border:1px solid #d0d7de; border-radius:6px;" />

Two things on that diagram are worth holding on to, because the rest of the notebook keeps
returning to them: **an `AGENTS.md`-style memory file is literally appended to the system
message**, wrapped in `<agent_memory>` tags &mdash; it is not a separate channel the model
consults. And **a skill costs you almost nothing until it is used**, because only its
frontmatter is indexed up front.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from dataclasses import dataclass

from dotenv import load_dotenv
load_dotenv(override=True)

from deepagents import create_deep_agent, FilesystemPermission
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from deepagents.backends.utils import create_file_data
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import MemorySaver

from util import print_exchange, show_store

model = "claude-haiku-4-5-20251001"

# The runtime context schema. Note this is NOT imported from deepagents — it is an
# ordinary dataclass you define yourself, and `context_schema=` below accepts it as-is.
# Every namespace lambda in this notebook reads its fields off `rt.context`.
@dataclass
class MemoryContext:
    """Runtime context: who is asking, and on whose behalf."""
    user_id: str
    org_id: str


alice = MemoryContext(user_id="user-alice", org_id="acme-corp")
bob = MemoryContext(user_id="user-bob", org_id="acme-corp")

# Who owns this copy? The namespace function

A route says *which backend* a file lives in. The `namespace` says *whose copy* it is.
`StoreBackend(namespace=...)` takes a callable that receives the runtime and returns a
namespace tuple &mdash; resolved fresh on every operation, so one compiled agent serves
every user without rebuilding.

Where that identity comes from depends on how the agent is running:

<table style="font-size:16px; border-collapse:collapse; width:100%;">
<thead>
<tr>
<th style="border:1px solid #999; padding:10px; text-align:left;">Source</th>
<th style="border:1px solid #999; padding:10px; text-align:left;">Looks like</th>
<th style="border:1px solid #999; padding:10px; text-align:left;">Available</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #999; padding:10px;">Runtime context</td>
<td style="border:1px solid #999; padding:10px;"><code style="color:#2b8a3e;">rt.context.user_id</code></td>
<td style="border:1px solid #999; padding:10px;">Anywhere &mdash; you pass it per invoke</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px;">Authenticated caller</td>
<td style="border:1px solid #999; padding:10px;"><code style="color:#2b8a3e;">rt.server_info.user.identity</code></td>
<td style="border:1px solid #999; padding:10px;">Only for agents running on LangSmith Deployments</td>
</tr>
</tbody>
</table>

Most documentation examples use `rt.server_info.user.identity`, because in a real
deployment the identity should come from the authenticated request rather than from
anything the caller can spoof. **Locally `rt.server_info` is `None`**, so that expression
raises `AttributeError` in a notebook. We use a `context_schema` instead, which works in
both places &mdash; and lets us play both Alice and Bob in the same kernel.

That schema is the `MemoryContext` dataclass defined in the setup cell above. It is not a
`deepagents` import &mdash; it is an ordinary dataclass of your own, and every namespace
function in this notebook reads its fields off `rt.context`:

<pre style="background:#f6f8fa; border:1px solid #d0d7de; border-radius:6px; padding:12px; font-size:14px; line-height:1.5; overflow:auto;"><code>StoreBackend(namespace=lambda rt: (<span style="color:#cf222e;">rt.context.user_id</span>, "memories"))
<span style="color:#6e7781;"># ...later, per call:</span>
agent.invoke(..., context=<span style="color:#cf222e;">MemoryContext(user_id="user-alice", org_id="acme-corp")</span>)</code></pre>

### What exactly is `rt`?

`rt` is just the lambda's parameter name &mdash; the object itself is a
<code style="font-size:15px; color:#2b8a3e;">langgraph.runtime.Runtime</code>, which
LangGraph builds once per run and `StoreBackend` resolves on **every** file operation
before handing it to your namespace factory. Resolving it per operation rather than at
build time is what lets a single compiled agent serve every user.

Its `context` field holds exactly the object you pass to `context=` on invoke &mdash; so
`rt.context.user_id` reads straight off the `MemoryContext` defined in the setup cell.

# Seeding the store

Application code, not the agent, puts the initial files in place. The store key has two
parts: the **namespace tuple** (whose copy) and the **key** (which file).

<div style="background:#fff8e1; border:2px solid #f0ad4e; border-radius:6px; padding:14px 18px; font-size:16px;">
<b>⚠️ The route prefix is stripped before the key reaches the backend.</b><br/><br/>
The agent reads <code style="color:#2b8a3e;">/memories/preferences.md</code>. The
<code style="color:#2b8a3e;">/memories/</code> route matches, so
<code style="color:#2b8a3e;">CompositeBackend</code> removes that prefix and asks its
<code style="color:#2b8a3e;">StoreBackend</code> for
<code style="color:#cf222e;">/preferences.md</code>. That stripped path is the real store
key.<br/><br/>
Seed <code style="color:#cf222e;">"/memories/preferences.md"</code> as the key and the
agent's read misses it &mdash; it reports the file does not exist, with no error to
explain why.
</div>

Routing is symmetric: what the agent asks for is `route prefix` + `store key`.

In [14]:
store = InMemoryStore()   # a platform-backed store in production

# Namespace = (user_id, "memories"); key = the path WITHOUT the /memories/ prefix.
store.put(("user-alice", "memories"), "/preferences.md", create_file_data(
    """## Preferences
- Likes concise bullet points
- Prefers Python examples
"""))

store.put(("user-bob", "memories"), "/preferences.md", create_file_data(
    """## Preferences
- Likes long, detailed explanations
- Prefers TypeScript examples
"""))

# Org-wide policy. Namespace = (org_id, "policies") — one copy for everyone in the org.
store.put(("acme-corp", "policies"), "/compliance.md", create_file_data(
    """## Compliance policies
- Never disclose internal pricing
- Always include a disclaimer on financial advice
"""))

# A per-user skill: procedural memory, read on demand rather than loaded up front.
store.put(("user-alice", "skills"), "/langgraph-docs/SKILL.md", create_file_data(
    """---
name: langgraph-docs
description: Look up LangGraph documentation to give accurate, current guidance.
---

Answer LangGraph questions by citing the official documentation at
https://docs.langchain.com rather than from memory.
"""))

show_store(store, ("user-alice", "memories"), title="Alice's memories — note the stripped key")

**Alice's memories — note the stripped key** — 1 file(s)

`/preferences.md`

<div class="highlight" style="background: #ffffff; background:#ffffff !important; color:#1a1a1a !important; border:1px solid #d0d7de; border-radius:6px; padding:12px 14px; margin:8px 0; overflow-x:auto;"><pre style="background:transparent !important; color:#1a1a1a !important; margin:0; line-height:1.45;; line-height: 125%;"><span></span>## Preferences
<span style="color: #A90D91">-</span> Likes concise bullet points
<span style="color: #A90D91">-</span> Prefers Python examples
</pre></div>


### Why each route gets its own namespace segment

The namespaces above end in `"memories"`, `"policies"`, `"skills"` &mdash; and that suffix is
load-bearing, for a reason that follows directly from prefix stripping.

Give `/memories/` and `/skills/` the same namespace (`(user_id,)`, as is tempting) and the
two routes stop being separate places. Both strip their prefix before hitting the store, so
`/memories/preferences.md` and `/skills/preferences.md` both resolve to key
`/preferences.md` in namespace `(user_id,)`. They are the same file. `ls` makes the
aliasing obvious once you look:

<pre style="background:#f6f8fa; border:1px solid #d0d7de; border-radius:6px; padding:12px; font-size:14px; line-height:1.5; overflow:auto;"><code><span style="color:#6e7781;"># one namespace shared by both routes — every file shows up under both</span>
ls /memories/  <span style="color:#6e7781;">→</span> ['/memories/langgraph-docs/', '/memories/preferences.md']
ls /skills/    <span style="color:#6e7781;">→</span> ['/skills/langgraph-docs/', '/skills/preferences.md']</code></pre>

The agent then discovers a "skill" called `preferences.md` and sees its own memory file
listed as procedural knowledge. Nothing errors; the routing table just quietly describes
one directory wearing two names.

Two routes may share a namespace only if you *want* them aliased. Otherwise give each one a
distinct namespace &mdash; a trailing segment is the cheapest way to do it, and it keeps the
store browsable besides.

# The agent

Four things wire the memory architecture together, and each does a different job:

- **`backend=CompositeBackend(...)`** &mdash; the routing table. Where bytes go.
- **`memory=[...]`** &mdash; files loaded into the system prompt at startup. The agent
  always knows these, without a tool call.
- **`skills=[...]`** &mdash; discovered at startup but only the *descriptions* are loaded;
  the body is read when a task matches.
- **`permissions=[...]`** &mdash; enforced by the harness, not by prompting.

The prompt is doing real work here too. Nothing in the framework makes an agent *save* a
preference &mdash; that behaviour comes entirely from the instructions below, and the
`edit_file` tool it already has.

In [4]:
MEMORY_INSTRUCTIONS = """You are a helpful coding assistant.

## Memory
You have a persistent memory file at /memories/preferences.md holding this user's durable
preferences: tools, languages, style, workflow.

- Apply what you find there without asking the user to repeat themselves.
- Watch for statements expressing a LASTING preference — "I prefer X over Y", "always do it
  this way", "don't do X". One-off, task-specific requests are NOT preferences.
- When you identify a durable preference, use edit_file to update
  /memories/preferences.md immediately. Add one short bullet under the relevant section,
  and update rather than duplicate an entry that already covers it.
- Never save credentials, secrets, or anything the user asks you to keep out of memory.
- Do not mention the file or the act of saving unless asked — just apply and update.

## Policies
/policies/compliance.md holds organization rules. Treat it as read-only background context
and never attempt to modify it.
"""

backend = CompositeBackend(
    # `default=` is the fallback: any path that matches NO route below is handled here.
    # StateBackend keeps files in the graph's state, so they are thread-scoped — they ride
    # the checkpointer and disappear with the thread. That is what we want for scratch
    # files and offloaded tool results, and it is why only the routed prefixes persist.
    default=StateBackend(),
    routes={
        # The trailing segment matters — see the note above on route collisions.
        "/memories/": StoreBackend(namespace=lambda rt: (rt.context.user_id, "memories")),
        "/policies/": StoreBackend(namespace=lambda rt: (rt.context.org_id, "policies")),
        "/skills/":   StoreBackend(namespace=lambda rt: (rt.context.user_id, "skills")),
    },
)

agent = create_deep_agent(
    model=model,
    system_prompt=MEMORY_INSTRUCTIONS,
    memory=["/memories/preferences.md", "/policies/compliance.md"],
    skills=["/skills/"],
    context_schema=MemoryContext,
    backend=backend,
    permissions=[
        # Enforced by the harness: org policy is changed by application code only.
        FilesystemPermission(operations=["write"], paths=["/policies/**"], mode="deny"),
    ],
    store=store,
    checkpointer=MemorySaver(),
)

# One filesystem, three destinations

From the agent's side none of this routing is visible. `ls /` returns a single tree; the
fact that three of those directories are durable store namespaces and everything else is
thread-scoped state is entirely the developer's business.

In [5]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Run ls on / and show me the raw result."}]},
    context=alice,
    config={"configurable": {"thread_id": "alice-ls"}},
)
print_exchange(result)

### Human

Run ls on / and show me the raw result.

### Ai

Here's the raw result from running `ls /`:

```
/memories/
/policies/
/skills/
```

The root directory contains three subdirectories:
- `/memories/` - for storing user preferences
- `/policies/` - for organization compliance rules
- `/skills/` - for the skills library with specialized capabilities

# Per-user isolation

`memory=[...]` puts `/memories/preferences.md` in the system prompt before the first turn,
so the agent starts every conversation already knowing the user's preferences.

The same compiled agent, the same file path, two different `context` values &mdash; and the
namespace function resolves to a different store namespace each time. Alice's preferences
cannot leak into Bob's conversation, because they were never in the same place.

In [6]:
question = "What are my stated preferences? Quote them back verbatim."

for who, context in (("Alice", alice), ("Bob", bob)):
    result = agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        context=context,
        config={"configurable": {"thread_id": f"prefs-{who}"}},
    )
    print(f"===== {who} =====")
    print(result["messages"][-1].content)
    print()

===== Alice =====
Your stated preferences are:

- Likes concise bullet points
- Prefers Python examples

===== Bob =====
Here are your stated preferences, verbatim:

- Likes long, detailed explanations
- Prefers TypeScript examples



# Why that isolation holds

Alice and Bob seeing different files is a good demo, but it invites the obvious question:
could Bob's agent simply *ask* for Alice's memory &mdash; talked into it by a crafty user, or
by instructions smuggled into something it reads?

No, and not because the model is well behaved. The isolation is structural: **the namespace
is a separate axis from the path.** The model only ever supplies a path string. The
namespace is computed from `rt.context` when the operation runs, and nothing the model
emits can reach it. Two layers sit in the way:

1. **Path validation** &mdash; defense in depth. `..` is rejected as a path component,
   checked again after normalization, and a *leading* `~` is rejected too. Bob's agent
   asking for `/memories/../../user-alice/…` just gets an ordinary tool error back:
   `Error: Path traversal not allowed: …`. Namespace components are separately restricted
   to a safe character set, so a crafted value cannot smuggle a wildcard into a store
   lookup.
2. **Namespace scoping** &mdash; the actual guarantee. Even a perfectly legal path is
   looked up *inside the caller's namespace*, so the same key resolves to a different file
   for a different caller.

Layer 1 is worth having, but do not lean on it: a `~` in the *middle* of a path is not
special-cased at all, it is simply a literal directory name. That is harmless precisely
because layer 2 does the real work &mdash; whatever key the path normalizes to, it is still
read inside `("user-bob", "memories")`.

Prompt injection cannot defeat this, because there is no phrasing that changes
`rt.context`. Doing that requires the *application* to pass a different `context=` &mdash;
which is why identity should come from an authenticated request in production, never from
anything the caller can set.

In [7]:
from deepagents.backends.utils import validate_path

# Layer 1 — path validation, before any backend sees the path.
print("Layer 1: path validation")
for path in [
    "/memories/preferences.md",                            # legitimate
    "/memories/../../user-alice/memories/preferences.md",  # climb out of the route
    "~/preferences.md",                                    # leading tilde
    "/memories/~/preferences.md",                          # tilde as a literal name
]:
    try:
        print(f"  allowed   {path:52} -> {validate_path(path)}")
    except ValueError as exc:
        print(f"  REJECTED  {path:52} -> {exc}")

# Layer 2 — the namespace, which no path string can influence. Note the last path
# above is allowed through, and still cannot escape: it resolves inside Bob's namespace.
print("\nLayer 2: one key, two namespaces")
for namespace in [("user-bob", "memories"), ("user-alice", "memories")]:
    content = store.get(namespace, "/preferences.md").value["content"]
    print(f"  {str(namespace):32} -> {content.splitlines()[1]}")

Layer 1: path validation
  allowed   /memories/preferences.md                             -> /memories/preferences.md
  REJECTED  /memories/../../user-alice/memories/preferences.md   -> Path traversal not allowed: /memories/../../user-alice/memories/preferences.md
  REJECTED  ~/preferences.md                                     -> Path traversal not allowed: ~/preferences.md
  allowed   /memories/~/preferences.md                           -> /memories/~/preferences.md

Layer 2: one key, two namespaces
  ('user-bob', 'memories')         -> - Likes long, detailed explanations
  ('user-alice', 'memories')       -> - Likes concise bullet points


# Read-only memory, enforced rather than requested

The prompt above asks the agent to treat `/policies/compliance.md` as read-only. That is a
*request*, and our agent honours it &mdash; which is exactly why it is a bad way to prove
anything. A well-behaved model declining politely looks identical to a guardrail working,
and tells you nothing about what happens when the model is confused, jailbroken, or
prompt-injected through the shared file itself.

So test the guardrail on its own. The agent below has the same
`FilesystemPermission(..., mode="deny")` rule, but **nothing in its prompt** about the file
being read-only. It has no reason not to comply with the user's request &mdash; and it
tries.

This matters most for exactly the memory that is *shared*. A per-user file can only harm
that user; an org-wide policy file that any user's agent can edit is a path from one
poisoned conversation into everyone else's system prompt.

In [8]:
unguarded = create_deep_agent(
    model=model,
    system_prompt="You are a helpful assistant. Do what the user asks.",  # no read-only rule
    context_schema=MemoryContext,
    backend=backend,
    permissions=[
        FilesystemPermission(operations=["write"], paths=["/policies/**"], mode="deny"),
    ],
    store=store,
)

result = unguarded.invoke(
    {"messages": [{"role": "user", "content":
        "Append the line '- No refunds under any circumstances' to /policies/compliance.md."}]},
    context=alice,
)

for message in result["messages"]:
    for call in getattr(message, "tool_calls", None) or []:
        print(f"→ {call['name']}({call['args'].get('file_path', '')})")
    if message.type == "tool":
        print(f"← {str(message.content).splitlines()[0][:90]}")

→ read_file(/policies/compliance.md)
← 1  ## Compliance policies
→ edit_file(/policies/compliance.md)
← Error: permission denied for write on /policies/compliance.md


The agent called `edit_file` and the harness rejected it &mdash; `Error: permission denied
for write on /policies/compliance.md`. The model's intent was irrelevant.

The store is the source of truth, not the model's account of what it did:

In [9]:
show_store(store, ("acme-corp", "policies"), title="Org policy after the attempted write")

**Org policy after the attempted write** — 1 file(s)

`/compliance.md`

<div class="highlight" style="background: #ffffff; background:#ffffff !important; color:#1a1a1a !important; border:1px solid #d0d7de; border-radius:6px; padding:12px 14px; margin:8px 0; overflow-x:auto;"><pre style="background:transparent !important; color:#1a1a1a !important; margin:0; line-height:1.45;; line-height: 125%;"><span></span>## Compliance policies
<span style="color: #A90D91">-</span> Never disclose internal pricing
<span style="color: #A90D91">-</span> Always include a disclaimer on financial advice
</pre></div>


# Skills come from the store too

`/skills/` is routed to the same kind of durable, user-scoped backend. Because skills use
progressive disclosure, only the frontmatter `name` and `description` are loaded at startup
&mdash; the body is read when a task actually calls for it.

Alice has a skill. Bob does not, and the difference costs nothing to arrange: it is the
same namespace function doing the same job.

In [10]:
for who, context in (("Alice", alice), ("Bob", bob)):
    result = agent.invoke(
        {"messages": [{"role": "user", "content":
            "List the names of any skills available to you. If none, say 'none'."}]},
        context=context,
        config={"configurable": {"thread_id": f"skills-{who}"}},
    )
    print(f"===== {who} =====")
    print(result["messages"][-1].content)
    print()

===== Alice =====
I have the following skill available:

- **langgraph-docs**: Look up LangGraph documentation to give accurate, current guidance.

===== Bob =====
None.



# The write path: a preference stated in passing

This is the whole point of the architecture. Alice mentions a preference *incidentally*,
while asking about something else. Nothing in the framework notices &mdash; the system
prompt is what tells the agent to watch for durable preferences and record them.

Watch what the agent says, then check what actually landed in the store.

In [11]:
result = agent.invoke(
    {"messages": [{"role": "user", "content":
        "I prefer uv over pip for Python packages. Anyway — what's the quickest way to "
        "read a CSV in Python?"}]},
    context=alice,
    config={"configurable": {"thread_id": "alice-csv"}},
)
print_exchange(result)

### Human

I prefer uv over pip for Python packages. Anyway — what's the quickest way to read a CSV in Python?

### Ai

**Quickest ways to read a CSV in Python:**

1. **pandas (most common)**
   ```python
   import pandas as pd
   df = pd.read_csv('file.csv')
   ```
   Install with: `uv pip install pandas`

2. **csv module (built-in, no install)**
   ```python
   import csv
   with open('file.csv') as f:
       reader = csv.DictReader(f)
       for row in reader:
           print(row)
   ```

3. **polars (fastest for large files)**
   ```python
   import polars as pl
   df = pl.read_csv('file.csv')
   ```
   Install with: `uv pip install polars`

**Pick based on your needs:**
- **Speed + simplicity**: pandas
- **No dependencies**: csv module
- **Large files**: polars
- **Data processing**: pandas or polars

Which use case fits your task?

The reply is about CSVs; the memory write happened quietly alongside it, exactly as the
prompt asked. Here is the file itself &mdash; and note the key is still the stripped
`/preferences.md`, now written by the agent rather than by us:

In [12]:
show_store(store, ("user-alice", "memories"), title="Alice's namespace after the write")

**Alice's namespace after the write** — 1 file(s)

`/preferences.md`

<div class="highlight" style="background: #ffffff; background:#ffffff !important; color:#1a1a1a !important; border:1px solid #d0d7de; border-radius:6px; padding:12px 14px; margin:8px 0; overflow-x:auto;"><pre style="background:transparent !important; color:#1a1a1a !important; margin:0; line-height:1.45;; line-height: 125%;"><span></span>## Preferences
<span style="color: #A90D91">-</span> Likes concise bullet points
<span style="color: #A90D91">-</span> Prefers Python examples
<span style="color: #A90D91">-</span> Uses uv for Python package management (not pip)
</pre></div>


# Durability: a brand-new thread

`StateBackend` would have lost that write the moment the thread ended. Because
`/memories/` routes to a `StoreBackend`, the file outlives the conversation &mdash; and
since it is listed in `memory=[...]`, it is back in the system prompt before Alice's next
conversation starts.

New `thread_id`, no checkpoint history, no mention of the earlier exchange:

In [13]:
result = agent.invoke(
    {"messages": [{"role": "user", "content":
        "Which package manager should I use for a new Python project? One line."}]},
    context=alice,
    config={"configurable": {"thread_id": "alice-tomorrow"}},
)
print_exchange(result)

# And Bob, in the same store, is untouched by any of it:
show_store(store, ("user-bob", "memories"), title="Bob's namespace — unchanged")

### Human

Which package manager should I use for a new Python project? One line.

### Ai

Use `uv` — it's fast, modern, and handles both dependency management and virtual environments seamlessly.

**Bob's namespace — unchanged** — 1 file(s)

`/preferences.md`

<div class="highlight" style="background: #ffffff; background:#ffffff !important; color:#1a1a1a !important; border:1px solid #d0d7de; border-radius:6px; padding:12px 14px; margin:8px 0; overflow-x:auto;"><pre style="background:transparent !important; color:#1a1a1a !important; margin:0; line-height:1.45;; line-height: 125%;"><span></span>## Preferences
<span style="color: #A90D91">-</span> Likes long, detailed explanations
<span style="color: #A90D91">-</span> Prefers TypeScript examples
</pre></div>


# Where this gets harder

The demo above is the happy path. Three things to plan for before shipping it:

- **Capture is prompt engineering, not a guarantee.** The model will miss preferences
  stated obliquely and occasionally save something trivial. Test the wording; treat it as a
  prompt you iterate on rather than a feature you switch on.
- **Concurrent writes are last-write-wins.** Rarely an issue for user-scoped memory, where
  one user usually has one active thread, but it is a real hazard for any file shared
  across concurrent sessions.
- **Consider consolidating out of band.** A separate agent that reviews conversations on a
  schedule and merges facts into memory is more reliable than asking the hot path to notice
  everything mid-task &mdash; and it can dedupe and rewrite entries rather than only
  appending.

Every write is an `edit_file` tool call, so all of it shows up in a LangSmith trace as an
ordinary, auditable step.

# Recap

- **`CompositeBackend` is a path-prefix router.** No semantic classification of memory
  happens anywhere &mdash; the meaning of `/memories/` versus `/policies/` lives in your
  path design and your prompt.
- **The route prefix is stripped before the key reaches the backend.** The agent's
  `/memories/preferences.md` is stored under `/preferences.md`. Seed the full path and the
  read silently misses.
- **`namespace` decides whose copy.** One compiled agent serves every user; the tuple is
  resolved per operation from runtime context (or, in a deployment, from the authenticated
  caller).
- **`memory=` loads eagerly, `skills=` loads on demand.** Both can be routed to durable,
  per-user storage.
- **Shared memory needs enforcement, not instructions.** `FilesystemPermission` with
  `mode="deny"` is what stops one user's conversation from rewriting everyone's context.
- **The agent only saves preferences because the prompt tells it to.** The tool it uses is
  the same `edit_file` it uses for everything else.